# Cafe Sales — Clean, Create, Encode (Beginner Notebook)

**Dataset:** `dirty_cafe_sales.csv` — 10,000 messy cafe transactions ([Kaggle: Cafe Sales — Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training))

Three simple steps, in order:

1. **Clean** the data first
2. **Create** a few simple new features
3. **Encode** the category columns into numbers

Every example here uses plain, short code — good for teaching the *idea* before showing fancier versions.

## 0. Load the data

In [9]:
import pandas as pd
import numpy as np
import datetime as dt

df = pd.read_csv("cleaned_data.csv")

print(df.shape)
df.head()

(10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [22]:

df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)
df["Transaction Date"].dtype


dtype('<M8[us]')

## 1. Clean

**Note:** this dataset uses three different ways to say "missing": a real blank, the text `"ERROR"`, and the text `"UNKNOWN"`. We turn all three into one proper missing value (`NaN`) so pandas treats them the same way.

In [11]:
# Step 1: turn "ERROR" and "UNKNOWN" into real missing values
df = df.replace("ERROR", np.nan)
df = df.replace("UNKNOWN", np.nan)

# How much is missing in each column?
df.isna().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

**Note:** `Quantity`, `Price Per Unit`, and `Total Spent` should be numbers, but they were loaded as text (because of the `"ERROR"`/`"UNKNOWN"` mixed in). Same with the date. We fix the data types next.

In [ ]:
# Step 2: fix data types
df["Quantity"] = pd.to_numeric(df["Quantity"])
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"])
df["Total Spent"] = pd.to_numeric(df["Total Spent"])
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

df.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

**Note:** for `Payment Method` and `Location`, missing just means "we don't know" — that's a fine value to keep, so we label it clearly instead of deleting the row.

In [ ]:
# Step 3: label missing categories instead of deleting them
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")

df["Payment Method"].value_counts()

Payment Method
Unknown           3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

**Note:** `Item`, `Quantity`, `Price Per Unit`, and `Transaction Date` are harder to guess — there's no safe default. For these, we simply drop the rows that are missing them.

In [ ]:
# Step 4: drop rows still missing something important
df = df.dropna(subset=["Item", "Quantity", "Price Per Unit", "Transaction Date"])

print("Rows remaining:", len(df))
df.isna().sum()

Rows remaining: 7773


Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent         391
Payment Method        0
Location              0
Transaction Date      0
dtype: int64

In [ ]:
# Step 5: fill any remaining Total Spent using Quantity x Price
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])

df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

The data is clean now: no missing values, correct data types. That's what we need before creating or encoding anything.

In [17]:
df.head()
df["Transaction Date"].dtype

<StringDtype(storage='python', na_value=nan)>

## 2. Create simple features

**Note:** a new feature is just a new column, usually built from columns we already have.

In [23]:
# Feature 1: day of the week the sale happened
df["Day_of_Week"] = df["Transaction Date"].dt.day_name()

df[["Transaction Date", "Day_of_Week"]].head()

,Transaction Date,Day_of_Week
0,2023-09-08,Friday
1,2023-05-16,Tuesday
2,2023-07-19,Wednesday
3,2023-04-27,Thursday
4,2023-06-11,Sunday


In [ ]:
# Feature 2: was it a weekend sale?
df["Is_Weekend"] = df["Day_of_Week"].isin(["Saturday", "Sunday"])

df[["Day_of_Week", "Is_Weekend"]].head()

,Day_of_Week,Is_Weekend
0,Friday,False
1,Tuesday,False
2,Wednesday,False
3,Thursday,False
4,Sunday,True


In [ ]:
# Feature 3: double-check Total Spent = Quantity x Price (should always be True)
#df["Total_Matches"] = df["Total Spent"] == (df["Quantity"] * df["Price Per Unit"])
df["Total_Matches"] = np.isclose(
    df["Total Spent"],
    df["Quantity"] * df["Price Per Unit"]
)

#df[df["Total_Matches"] == False][
 #   ["Total_Matches", "Total Spent", "Quantity", "Price Per Unit"]
#]
df["Expected_Total"] = df["Quantity"] * df["Price Per Unit"]
df.loc[~df["Total_Matches"],
       ["Total Spent", "Quantity", "Price Per Unit", "Expected_Total"]]

In [34]:
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]
df[df["Total_Matches"] == False][
   ["Total_Matches", "Total Spent", "Quantity", "Price Per Unit"]
]


,Total_Matches,Total Spent,Quantity,Price Per Unit


In [ ]:
# Feature 4: simple price category - cheap, medium, or expensive
def price_group(price):
    if price <= 2:
        return "Cheap"
    elif price <= 4:
        return "Medium"
    else:
        return "Expensive"

df["Price_Group"] = df["Price Per Unit"].apply(price_group)

df["Price_Group"].value_counts()

Price_Group
Medium       3929
Cheap        2859
Expensive     985
Name: count, dtype: int64

In [ ]:
# Feature 5: how many items were bought at once - small or large order
df["Large_Order"] = df["Quantity"] >= 4

df["Large_Order"].value_counts()

Large_Order
False    4616
True     3157
Name: count, dtype: int64

Five new columns, five short pieces of code. That's the whole idea behind feature creation: **ask a simple question about the row, and turn the answer into a column.**

## 3. Encode the categories

**Note:** models only understand numbers, not text like `"Coffee"` or `"Cash"`. Encoding turns each category into a number.

### 3.1 One-hot encoding — the simple, safe default

`pd.get_dummies()` makes one new 0/1 column per category.

In [36]:
item_encoded = pd.get_dummies(df["Item"], prefix="Item")

item_encoded.head()

,Item_Cake,Item_Coffee,Item_Cookie,Item_Juice,Item_Salad,Item_Sandwich,Item_Smoothie,Item_Tea
0,False,True,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False
2,False,False,True,False,False,False,False,False
3,False,False,False,False,True,False,False,False
4,False,True,False,False,False,False,False,False


In [ ]:
payment_encoded = pd.get_dummies(df["Payment Method"], prefix="Pay")

payment_encoded.head()

,Pay_Cash,Pay_Credit Card,Pay_Digital Wallet,Pay_Unknown
0,False,True,False,False
1,True,False,False,False
2,False,True,False,False
3,False,False,False,True
4,False,False,True,False


### 3.2 Simple mapping — when you want to choose the numbers yourself

Sometimes it's easier to just write your own number for each category using `.map()`.

In [ ]:
# Turn True/False columns into 1/0 - most models prefer plain numbers
df["Is_Weekend"] = df["Is_Weekend"].astype(int)
df["Large_Order"] = df["Large_Order"].astype(int)

df[["Is_Weekend", "Large_Order"]].head()

,Is_Weekend,Large_Order
0,0,0
1,0,1
2,0,1
3,0,0
4,1,0


In [ ]:
# Location only has two values, so a simple map works well
location_map = {"In-store": 1, "Takeaway": 0, "Unknown": -1}
df["Location_Code"] = df["Location"].map(location_map)

df[["Location", "Location_Code"]].head()

,Location,Location_Code
0,Takeaway,0
1,In-store,1
2,In-store,1
3,Unknown,-1
4,In-store,1


In [ ]:
# Price_Group has a natural order (Cheap < Medium < Expensive), so we map it to ordered numbers
price_group_map = {"Cheap": 0, "Medium": 1, "Expensive": 2}
df["Price_Group_Code"] = df["Price_Group"].map(price_group_map)

df[["Price_Group", "Price_Group_Code"]].head()

,Price_Group,Price_Group_Code
0,Cheap,0
1,Medium,1
2,Cheap,0
3,Expensive,2
4,Cheap,0


### 3.3 Put the encoded columns together

In [ ]:
final_df = pd.concat([
    df[["Quantity", "Price Per Unit", "Total Spent", "Is_Weekend", "Large_Order",
        "Location_Code", "Price_Group_Code"]],
    item_encoded,
    payment_encoded,
], axis=1)

print(final_df.shape)
final_df.head()

(7773, 19)


,Quantity,Price Per Unit,Total Spent,Is_Weekend,Large_Order,Location_Code,Price_Group_Code,Item_Cake,Item_Coffee,Item_Cookie,Item_Juice,Item_Salad,Item_Sandwich,Item_Smoothie,Item_Tea,Pay_Cash,Pay_Credit Card,Pay_Digital Wallet,Pay_Unknown
0,2.0,2.0,4.0,0,0,0,0,False,True,False,False,False,False,False,False,False,True,False,False
1,4.0,3.0,12.0,0,1,1,1,True,False,False,False,False,False,False,False,True,False,False,False
2,4.0,1.0,4.0,0,1,1,0,False,False,True,False,False,False,False,False,False,True,False,False
3,2.0,5.0,10.0,0,0,-1,2,False,False,False,False,True,False,False,False,False,False,False,True
4,2.0,2.0,4.0,1,0,1,0,False,True,False,False,False,False,False,False,False,False,True,False


## Recap

| Step | What we did |
|---|---|
| Clean | Fixed the fake missing values, fixed data types, filled or dropped what was left |
| Create | Added `Day_of_Week`, `Is_Weekend`, `Total_Matches`, `Price_Group`, `Large_Order` |
| Encode | One-hot encoded `Item` and `Payment Method`; mapped `Location` and `Price_Group` to numbers by hand |

### Try it yourself
1. Create a new feature `Is_Coffee` that is `True` when `Item` is `"Coffee"`.
2. One-hot encode `Day_of_Week` the same way we did `Item`.
3. Make your own `.map()` for `Payment Method` instead of one-hot encoding it.

In [37]:
df['Is_Coffee'] = df['Item'] == 'Coffee'
df['Is_Coffee'].head()

0     True
1    False
2    False
3    False
4     True
Name: Is_Coffee, dtype: bool

In [ ]:
df['Is_Coffee'].unique()

array([ True, False])

In [39]:
encode_day = pd.get_dummies(df['Day_of_Week'], prefix='Day') 
encode_day.head()

,Day_Friday,Day_Monday,Day_Saturday,Day_Sunday,Day_Thursday,Day_Tuesday,Day_Wednesday
0,True,False,False,False,False,False,False
1,False,False,False,False,False,True,False
2,False,False,False,False,False,False,True
3,False,False,False,False,True,False,False
4,False,False,False,True,False,False,False


In [40]:
df['Payment Method'].unique()

<StringArray>
['Credit Card', 'Cash', 'Unknown', 'Digital Wallet']
Length: 4, dtype: str

In [42]:
payment_map ={
    'Cash': 0,
    'Credit Card': 1,
    'Digital Wallet': 2,
}
df['Payment_code'] = df['Payment Method'].map(payment_map)
df[['Payment Method', 'Payment_code']].head()

,Payment Method,Payment_code
0,Credit Card,1.0
1,Cash,0.0
2,Credit Card,1.0
3,Unknown,NaN
4,Digital Wallet,2.0
